# NB13 · Modules, Files, JSON & CSV

`intermediate/` · notebook 13 of 32 · 36 questions + a mini-project

Until now every value lived in memory and vanished when the cell finished.
This notebook is about the outside world: splitting code into modules,
reading and writing files, and the two text formats you'll meet most,
JSON and CSV.

**Your real files are safe.** Most starting code makes a throwaway folder
with `tempfile.TemporaryDirectory()` and calls it `folder`. Every file in
these questions lives there, and Python deletes the folder for you (when
`tmp` is cleaned up, when you rerun the cell, or when the kernel stops).
Q05 shows the `with` form, which deletes it the moment the block ends.

**Writing your own modules.** A few questions write a small `.py` file into
that folder and add the folder to `sys.path` so `import` can find it.
Python imports each module only once per session, so if you change the
module's text and rerun, restart the kernel (or use `importlib.reload`).

Always pass `encoding="utf-8"` when you open text files. Without it, Python
uses your system's default, which differs between Windows, macOS and Linux.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB12: exceptions

## What you'll practise

- import styles: import x, from x import y, import x as y
- Writing your own module; if __name__ == '__main__'; packages and __init__.py
- pathlib: Path, /, name, stem, suffix, exists, glob, read_text, write_text
- open() modes r, w, a, x; encoding='utf-8'; using with
- Reading line by line and processing as you go
- json: dumps, loads, dump, load, indent, sort_keys
- csv: reader, writer, DictReader, DictWriter, newline=''
- tempfile, so exercises never touch your real files
- configparser and xml.etree.ElementTree (short tour)
- pickle and shelve, and why you never unpickle data you don't trust
- OSError family: FileNotFoundError, PermissionError, errno
- os and glob basics; dir(); sys.path; relative imports; import *; __pycache__

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB13-Q01 · L1 · open() and with · Write

Write the three lines `milk`, `eggs` and `bread` to `folder / "shopping.txt"`,
one per line. Then open the file again, read it, and print its contents.

```text
milk
eggs
bread
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)
```

<details><summary><b>Hint</b></summary>

`open(path, "w", encoding="utf-8")` opens for writing. `f.write()` does not add a newline for you, so put `\n` at the end of each line. `f.read()` gives the whole file as one string.

</details>

<details><summary><b>Expected output</b></summary>

```text
milk
eggs
bread
```

</details>

<details><summary><b>Solution</b></summary>

```python
path = folder / "shopping.txt"
with open(path, "w", encoding="utf-8") as f:
    f.write("milk\n")
    f.write("eggs\n")
    f.write("bread\n")

with open(path, encoding="utf-8") as f:
    print(f.read(), end="")
```

`with open(...) as f:` opens the file and *guarantees* it is closed when the block ends, even if an error happens inside. Mode `"w"` means write; leaving the mode out means `"r"` (read). The file already ends with `\n`, so `end=""` stops `print()` adding a blank line after it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
path = folder / "shopping.txt"
items = ["milk", "eggs", "bread"]
path.write_text("\n".join(items) + "\n", encoding="utf-8")
print(path.read_text(encoding="utf-8"), end="")
```

For small files, `Path.write_text` and `Path.read_text` open, write or read, and close in one call. Seniors keep `open()` for files they process piece by piece, and keep the data in a list so adding a fourth item is a one-word change.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

# your code here


### NB13-Q02 · L1 · Import styles · Write

Use three import styles in one cell: `import math`, `from math import sqrt`,
and `import math as m`. Then print `floor(7.8)` using the first, `sqrt(81)`
using the second, and `pi` rounded to 2 places using the third, on one line.

```text
7 9.0 3.14
```

<details><summary><b>Hint</b></summary>

After `import math` you write `math.floor`. After `from math import sqrt` you write plain `sqrt`. After `import math as m` you write `m.pi`.

</details>

<details><summary><b>Expected output</b></summary>

```text
7 9.0 3.14
```

</details>

<details><summary><b>Solution</b></summary>

```python
import math
from math import sqrt
import math as m

print(math.floor(7.8), sqrt(81), round(m.pi, 2))
```

`import math` gives you one name, `math`, and you reach inside it with a dot. `from math import sqrt` copies one name straight into your code. `import math as m` is the same module under a shorter name. All three point at the *same* module object; Python loads it only once.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from math import floor, pi, sqrt

print(floor(7.8), sqrt(81), round(pi, 2))
```

Pick one style per module and stick with it. `as` is for long or conventional names (`import numpy as np`), not for shortening `math`. PEP 8 also says: imports at the top, standard library first, one module per line, and names in alphabetical order makes them easy to scan.

</details>

In [ ]:
# your code here


### NB13-Q03 · L1 · pathlib · Predict

`pathlib.Path` splits a path into its parts. Predict the output.

**What does this print?**

```python
from pathlib import Path

report = Path("reports") / "2024" / "sales.final.csv"
print(report.as_posix())
print(report.name, report.stem, report.suffix)
print(report.parent.name)
```

<details><summary><b>Hint</b></summary>

`/` joins path pieces. `suffix` is only the *last* dot part. `stem` is the name without that last suffix.

</details>

<details><summary><b>Expected output</b></summary>

```text
reports/2024/sales.final.csv
sales.final.csv sales.final .csv
2024
```

</details>

<details><summary><b>Solution</b></summary>

`/` on a `Path` joins pieces with the right separator for your system. `name` is the last part, `sales.final.csv`. `suffix` is only the last extension, `.csv`, so `stem` keeps the earlier dot: `sales.final`. `parent` is the folder `reports/2024`, whose own name is `2024`. `as_posix()` prints with `/` even on Windows, which uses `\`.

</details>

<details><summary><b>Senior dev solution</b></summary>

`report.suffixes` gives every extension (`['.final', '.csv']`), handy for files like `backup.tar.gz`. Seniors build paths with `/` and never glue strings together with `"/"`, so the code works on every operating system.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB13-Q04 · L1 · json · Write

Print `book` as JSON text with `json.dumps`. Look closely at what happens to
`True` and `None`.

**Starting code (already in the cell below):**

```python
import json

book = {"title": "Dune", "year": 1965, "in_stock": True, "isbn": None,
        "tags": ["sci-fi", "classic"]}
```

<details><summary><b>Hint</b></summary>

`json.dumps(value)` (the `s` is for *string*) turns a Python value into JSON text.

</details>

<details><summary><b>Expected output</b></summary>

```text
{"title": "Dune", "year": 1965, "in_stock": true, "isbn": null, "tags": ["sci-fi", "classic"]}
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(json.dumps(book))
```

JSON is a text format shared by almost every language. It looks like a Python dict, with differences: strings always use double quotes, `True` becomes `true`, `False` becomes `false`, and `None` becomes `null`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
json_text = json.dumps(book)
print(json_text)
```

Compare it with `print(book)`: they look alike, but only the JSON text can be read by JavaScript, Go or any other language. Adding `indent=2` spreads it over several lines for humans; seniors use that for files people open and leave it out for data sent between programs.

</details>

In [ ]:
import json

book = {"title": "Dune", "year": 1965, "in_stock": True, "isbn": None,
        "tags": ["sci-fi", "classic"]}

# your code here


### NB13-Q05 · L1 · tempfile · Write

Create a temporary folder with `with tempfile.TemporaryDirectory() as tmp:`.
Inside the block, write `"hello"` to `notes.txt` in that folder and print
whether the file exists. After the block, print whether it still exists.

```text
True
False
```

<details><summary><b>Hint</b></summary>

`tmp` is the folder's path as a string; `Path(tmp) / "notes.txt"` is the file. `.exists()` returns a bool. The second check goes *after* the `with` block, with no indent.

</details>

<details><summary><b>Expected output</b></summary>

```text
True
False
```

</details>

<details><summary><b>Solution</b></summary>

```python
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    notes = Path(tmp) / "notes.txt"
    notes.write_text("hello", encoding="utf-8")
    print(notes.exists())
print(notes.exists())
```

`TemporaryDirectory()` makes a fresh, empty folder somewhere safe (in the system's temp area). Used with `with`, the folder and everything in it is deleted when the block ends. The name `notes` still exists after the block; the *file* it points to doesn't.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    notes = Path(tmp) / "notes.txt"
    notes.write_text("hello", encoding="utf-8")
    print(notes.exists())
print(notes.exists())
```

Same code. Seniors use temporary folders in tests for exactly this reason: each test gets a clean folder, and nothing is left behind even when the test fails halfway, because `with` cleans up on errors too.

</details>

In [ ]:
# your code here


---

## L2 · Easy

### NB13-Q06 · L2 · json · Predict

A dict goes to JSON and back. Predict the output.

**What does this print?**

```python
import json

original = {1: (1, 2)}
back = json.loads(json.dumps(original))
print(back)
print(back == original)
```

<details><summary><b>Hint</b></summary>

JSON object keys can only be strings, and JSON has no tuple type.

</details>

<details><summary><b>Expected output</b></summary>

```text
{'1': [1, 2]}
False
```

</details>

<details><summary><b>Solution</b></summary>

JSON keys are always strings, so the key `1` is written as `"1"`. JSON has arrays but no tuples, so `(1, 2)` becomes a list. Loading it back gives `{'1': [1, 2]}`, which is *not* equal to the original. A JSON round trip only preserves `str`, `int`, `float`, `bool`, `None`, `list` and dicts with string keys.

</details>

<details><summary><b>Senior dev solution</b></summary>

When data must survive a JSON round trip, seniors design it that way from the start: string keys and lists. If you need int keys back, convert them on load (`{int(k): v for k, v in data.items()}`), as Q29 does.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB13-Q07 · L2 · Reading line by line · Write

Loop over the open file line by line and count the lines and the words. A
blank line still counts as a line.

```text
lines: 5 words: 14
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

path = folder / "poem.txt"
path.write_text("The fog comes\non little cat feet.\n\nIt sits looking\nover harbor and city\n",
                encoding="utf-8")
```

<details><summary><b>Hint</b></summary>

`for line in f:` gives one line at a time (with its `\n`). `line.split()` with no argument splits on any whitespace and ignores the newline.

</details>

<details><summary><b>Expected output</b></summary>

```text
lines: 5 words: 14
```

</details>

<details><summary><b>Solution</b></summary>

```python
lines = 0
words = 0
with open(path, encoding="utf-8") as f:
    for line in f:
        lines += 1
        words += len(line.split())
print("lines:", lines, "words:", words)
```

An open text file is something you can loop over: each step gives the next line. The blank line is `"\n"`, which counts as a line but splits into zero words. Only one line is in memory at a time.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
with open(path, encoding="utf-8") as f:
    counts = [len(line.split()) for line in f]
print(f"lines: {len(counts)} words: {sum(counts)}")
```

Looping over the file instead of calling `f.read()` means a 10 GB log file uses as little memory as a 10 line one. Seniors process files as they stream past whenever they can.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

path = folder / "poem.txt"
path.write_text("The fog comes\non little cat feet.\n\nIt sits looking\nover harbor and city\n",
                encoding="utf-8")

# your code here


### NB13-Q08 · L2 · File modes · Predict

Four writes to the same file, with two different modes. Predict the output.

**What does this print?**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "log.txt"

with open(path, "w", encoding="utf-8") as f:
    f.write("first\n")
with open(path, "a", encoding="utf-8") as f:
    f.write("second\n")
with open(path, "w", encoding="utf-8") as f:
    f.write("third\n")
with open(path, "a", encoding="utf-8") as f:
    f.write("fourth\n")
print(path.read_text(encoding="utf-8"), end="")
```

<details><summary><b>Hint</b></summary>

`"w"` empties the file the moment it opens. `"a"` adds to the end.

</details>

<details><summary><b>Expected output</b></summary>

```text
third
fourth
```

</details>

<details><summary><b>Solution</b></summary>

Mode `"w"` *wipes* the file as soon as it is opened, before you write anything. Mode `"a"` (append) keeps what's there and writes at the end. So `first` and `second` are destroyed by the second `"w"`, leaving `third` and then `fourth`.

</details>

<details><summary><b>Senior dev solution</b></summary>

This is how real logs get lost: one `"w"` where an `"a"` was meant. When a file must not be overwritten, seniors use mode `"x"`, which refuses if the file already exists (Q10).

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB13-Q09 · L2 · csv · Write

Write `rows` to `path` with `csv.writer`, then print the file's text.

```text
name,score
Ada,91
Linus,78
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

rows = [["name", "score"], ["Ada", 91], ["Linus", 78]]
path = folder / "scores.csv"
```

<details><summary><b>Hint</b></summary>

Open the file with `newline=""` and `encoding="utf-8"`, make `writer = csv.writer(f)`, and call `writer.writerow(row)` for each row.

</details>

<details><summary><b>Expected output</b></summary>

```text
name,score
Ada,91
Linus,78
```

</details>

<details><summary><b>Solution</b></summary>

```python
with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    for row in rows:
        writer.writerow(row)
print(path.read_text(encoding="utf-8"), end="")
```

`csv.writer` turns each list into one line, adding commas (and quotes around values that contain commas). It writes numbers with `str()`. `newline=""` lets the csv module control line endings itself; Q36 shows what goes wrong without it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
with open(path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(rows)
print(path.read_text(encoding="utf-8"), end="")
```

`writerows` writes a whole list of rows in one call. Seniors use the csv module even for "simple" files, because the first value with a comma in it (`"Smith, John"`) breaks hand-made `",".join(...)` code.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

rows = [["name", "score"], ["Ada", 91], ["Linus", 78]]
path = folder / "scores.csv"

# your code here


### NB13-Q10 · L2 · File modes · Error

Mode `"x"` means *create*. Which error does this raise?

**Which error does this raise?**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

path = folder / "invoice_001.txt"
path.write_text("Total: 40.00", encoding="utf-8")

with open(path, "x", encoding="utf-8") as f:
    f.write("Total: 99.00")
```

<details><summary><b>Hint</b></summary>

The file was created on the line before. What should "create" do if it's already there?

</details>

<details><summary><b>Expected output</b></summary>

```text
FileExistsError
```

</details>

<details><summary><b>Solution</b></summary>

`FileExistsError`: mode `"x"` creates a new file and refuses to touch one that already exists. It's a safety catch: the first invoice survives instead of being silently wiped, which is what `"w"` would do.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

path = folder / "invoice_001.txt"
path.write_text("Total: 40.00", encoding="utf-8")

try:
    with open(path, "x", encoding="utf-8") as f:
        f.write("Total: 99.00")
except FileExistsError:
    print("Invoice already exists; not overwriting it")
```

Seniors reach for `"x"` whenever overwriting would be a disaster (invoices, exports, backups), and catch `FileExistsError` to tell the user. Checking `exists()` first and then writing has a gap where another program could create the file in between.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB13-Q11 · L2 · json · Write

Save `settings` to `path` with `json.dump`. Then load it back into a new
variable `loaded` with `json.load`, and print `loaded["font_size"] + 2` and
whether `loaded == settings`.

```text
16 True
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import json

settings = {"theme": "dark", "font_size": 14, "recent": ["a.txt", "b.txt"]}
path = folder / "settings.json"
```

<details><summary><b>Hint</b></summary>

`json.dump(value, f)` writes to an open file; `json.load(f)` reads from one. Open the file for writing the first time and for reading the second.

</details>

<details><summary><b>Expected output</b></summary>

```text
16 True
```

</details>

<details><summary><b>Solution</b></summary>

```python
with open(path, "w", encoding="utf-8") as f:
    json.dump(settings, f)

with open(path, encoding="utf-8") as f:
    loaded = json.load(f)

print(loaded["font_size"] + 2, loaded == settings)
```

`dump`/`load` work with files; `dumps`/`loads` (with an `s`) work with strings. Here every key is a string and every value is a JSON type, so the round trip is perfect and the number comes back as an `int`, ready for maths.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
path.write_text(json.dumps(settings, indent=2), encoding="utf-8")
loaded = json.loads(path.read_text(encoding="utf-8"))
print(loaded["font_size"] + 2, loaded == settings)
```

Both ways are fine. Seniors write settings with `indent=2` because people edit settings files by hand. Remember the `s`: `dumps` is *dump string*.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import json

settings = {"theme": "dark", "font_size": 14, "recent": ["a.txt", "b.txt"]}
path = folder / "settings.json"

# your code here


### NB13-Q12 · L2 · os basics · Write

Use `os.listdir(folder)` to print the names of the text files in `folder`,
in alphabetical order, one per line. A text file ends in `.txt` in **any**
case, so `readme.TXT` counts.

```text
notes.txt
readme.TXT
todo.txt
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import os

for name in ["notes.txt", "photo.jpg", "todo.txt", "data.csv", "readme.TXT"]:
    (folder / name).write_text("x", encoding="utf-8")
```

<details><summary><b>Hint</b></summary>

`os.listdir` returns a list of names in no particular order, so sort it. Lower the name before checking how it ends.

</details>

<details><summary><b>Expected output</b></summary>

```text
notes.txt
readme.TXT
todo.txt
```

</details>

<details><summary><b>Solution</b></summary>

```python
for name in sorted(os.listdir(folder)):
    if name.lower().endswith(".txt"):
        print(name)
```

`os.listdir` gives the bare names (not full paths) in whatever order the operating system returns them, which differs between systems. `sorted()` makes the output predictable. `.lower()` makes `.TXT` match too, while still printing the original name.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
for path in sorted(folder.iterdir()):
    if path.suffix.lower() == ".txt":
        print(path.name)
```

`Path.iterdir()` gives full `Path` objects, so `.suffix` and `.name` are ready to use. Seniors avoid `glob("*.txt")` here: it is case-sensitive on Linux but not on Windows, so the same code would find different files on different machines.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import os

for name in ["notes.txt", "photo.jpg", "todo.txt", "data.csv", "readme.TXT"]:
    (folder / name).write_text("x", encoding="utf-8")

# your code here


### NB13-Q13 · L2 · OSError family · Write

Read the high score from `path` as a whole number. If the file doesn't exist
yet (first time the game runs), use `0` instead. Print the result.

```text
High score: 0
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

path = folder / "high_score.txt"  # this file does not exist yet
```

<details><summary><b>Hint</b></summary>

Try to read it, and catch the specific error that a missing file raises.

</details>

<details><summary><b>Expected output</b></summary>

```text
High score: 0
```

</details>

<details><summary><b>Solution</b></summary>

```python
try:
    high_score = int(path.read_text(encoding="utf-8"))
except FileNotFoundError:
    high_score = 0
print("High score:", high_score)
```

Reading a missing file raises `FileNotFoundError`. Catching exactly that error (not every error) gives a sensible default on the first run, while a real problem, like the file containing `"abc"`, still shows up as a `ValueError`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
try:
    high_score = int(path.read_text(encoding="utf-8"))
except FileNotFoundError:
    high_score = 0
print(f"High score: {high_score}")
```

This is EAFP from NB12: try it and handle the failure. The LBYL version, `if path.exists():`, has a gap between the check and the read where the file could vanish. With files, seniors almost always try first.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

path = folder / "high_score.txt"  # this file does not exist yet

# your code here


### NB13-Q14 · L2 · csv · Fix

This should print how many pieces of fruit are in stock (`7`), but it prints
`34`. Fix it.

**Buggy code (copied into the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

path = folder / "stock.csv"
path.write_text("item,qty\napples,3\npears,4\n", encoding="utf-8")

with open(path, newline="", encoding="utf-8") as f:
    rows = list(csv.reader(f))

total = rows[1][1] + rows[2][1]
print("Fruit in stock:", total)
```

<details><summary><b>Hint</b></summary>

What type is every value that comes out of `csv.reader`?

</details>

<details><summary><b>Expected output</b></summary>

```text
Fruit in stock: 7
```

</details>

<details><summary><b>Solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

path = folder / "stock.csv"
path.write_text("item,qty\napples,3\npears,4\n", encoding="utf-8")

with open(path, newline="", encoding="utf-8") as f:
    rows = list(csv.reader(f))

total = int(rows[1][1]) + int(rows[2][1])
print("Fruit in stock:", total)
```

A CSV file is plain text, so `csv.reader` gives every value as a `str`. `"3" + "4"` joins the text into `"34"`. Converting each value with `int()` before adding gives `7`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

path = folder / "stock.csv"
path.write_text("item,qty\napples,3\npears,4\n", encoding="utf-8")

with open(path, newline="", encoding="utf-8") as f:
    total = sum(int(row["qty"]) for row in csv.DictReader(f))
print("Fruit in stock:", total)
```

`csv.DictReader` (Q16) lets you write `row["qty"]` instead of `row[1]`, so adding a column doesn't break the code. Seniors convert values the moment they're read, so no string ever reaches the maths.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

path = folder / "stock.csv"
path.write_text("item,qty\napples,3\npears,4\n", encoding="utf-8")

with open(path, newline="", encoding="utf-8") as f:
    rows = list(csv.reader(f))

total = rows[1][1] + rows[2][1]
print("Fruit in stock:", total)


### NB13-Q15 · L2 · open() and with · Error

Which error does this raise?

**Which error does this raise?**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

path = folder / "diary.txt"
with open(path, "w", encoding="utf-8") as f:
    f.write("Monday: rain\n")
f.write("Tuesday: sun\n")
```

<details><summary><b>Hint</b></summary>

Look at the indentation of the last line. What has `with` done by then?

</details>

<details><summary><b>Expected output</b></summary>

```text
ValueError
```

</details>

<details><summary><b>Solution</b></summary>

`ValueError`: when the `with` block ends, the file is closed. `f` still exists as a name, but writing to a closed file is not allowed, so Python raises `ValueError` ("I/O operation on closed file"). The fix is to indent the last write so it's inside the block.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

path = folder / "diary.txt"
with open(path, "w", encoding="utf-8") as f:
    f.write("Monday: rain\n")
    f.write("Tuesday: sun\n")
print(path.read_text(encoding="utf-8"), end="")
```

Seniors keep `with` blocks short and do *all* the file work inside them. If you need the data later, read it into a variable inside the block and use the variable afterwards.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

---

## L3 · Intermediate

### NB13-Q16 · L3 · csv · Write

Read `grades.csv` with `csv.DictReader` into a list of dicts. Then print the
average of every numeric column (every column except `student`), with one
decimal place.

```text
maths: 82.3
physics: 84.7
art: 84.3
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

path = folder / "grades.csv"
path.write_text(
    "student,maths,physics,art\n"
    "Ada,90,85,70\n"
    "Linus,72,91,88\n"
    "Grace,85,78,95\n",
    encoding="utf-8",
)
```

<details><summary><b>Hint</b></summary>

`list(csv.DictReader(f))` gives one dict per row, keyed by the header. Loop over the keys of the first row, skip `student`, and add up that column with `int()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
maths: 82.3
physics: 84.7
art: 84.3
```

</details>

<details><summary><b>Solution</b></summary>

```python
with open(path, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

for column in rows[0]:
    if column == "student":
        continue
    total = 0
    for row in rows:
        total += int(row[column])
    print(f"{column}: {total / len(rows):.1f}")
```

`DictReader` uses the first line as the keys, so each row looks like `{'student': 'Ada', 'maths': '90', ...}`. Dicts keep insertion order, so looping over `rows[0]` visits the columns in file order. Every value is text, so `int()` is needed before adding.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
with open(path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    numeric_columns = [name for name in reader.fieldnames if name != "student"]

for column in numeric_columns:
    average = sum(int(row[column]) for row in rows) / len(rows)
    print(f"{column}: {average:.1f}")
```

`reader.fieldnames` is the header row, and it exists even when the file has no data rows (where `rows[0]` would raise `IndexError`). For real number crunching on CSVs, seniors use `statistics.mean` (NB15) or a library like pandas.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

path = folder / "grades.csv"
path.write_text(
    "student,maths,physics,art\n"
    "Ada,90,85,70\n"
    "Linus,72,91,88\n"
    "Grace,85,78,95\n",
    encoding="utf-8",
)

# your code here


### NB13-Q17 · L3 · pathlib · Write

Print the `.txt` files in `folder`, smallest first, with their size in
bytes. Ignore other file types.

```text
b_todo.txt 15
e_draft.txt 40
a_notes.txt 120
c_log.txt 900
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

files = {"a_notes.txt": "x" * 120, "b_todo.txt": "x" * 15, "c_log.txt": "x" * 900,
         "d_photo.png": "x" * 5000, "e_draft.txt": "x" * 40}
for name, text in files.items():
    (folder / name).write_text(text, encoding="utf-8")
```

<details><summary><b>Hint</b></summary>

`folder.glob("*.txt")` finds the files. `path.stat().st_size` is a file's size in bytes. Sorting a list of `(size, name)` tuples sorts by size first.

</details>

<details><summary><b>Expected output</b></summary>

```text
b_todo.txt 15
e_draft.txt 40
a_notes.txt 120
c_log.txt 900
```

</details>

<details><summary><b>Solution</b></summary>

```python
sizes = []
for path in folder.glob("*.txt"):
    sizes.append((path.stat().st_size, path.name))

for size, name in sorted(sizes):
    print(name, size)
```

`glob("*.txt")` matches every name ending in `.txt`. `stat()` asks the operating system about the file, and `st_size` is its size. Tuples sort by their first item, so putting the size first sorts by size.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
by_size = sorted(folder.glob("*.txt"), key=lambda path: path.stat().st_size)
for path in by_size:
    print(path.name, path.stat().st_size)
```

`key=` (NB10) sorts the paths themselves by size, with no tuple building. This calls `stat()` twice per file, which is fine for a handful; for thousands, the tuple version that asks once is faster.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

files = {"a_notes.txt": "x" * 120, "b_todo.txt": "x" * 15, "c_log.txt": "x" * 900,
         "d_photo.png": "x" * 5000, "e_draft.txt": "x" * 40}
for name, text in files.items():
    (folder / name).write_text(text, encoding="utf-8")

# your code here


### NB13-Q18 · L3 · File modes · Fix

This should save all three orders, one per line, but only the last one
survives. Fix it.

```text
A101 coffee
A102 tea
A103 cake
```

**Buggy code (copied into the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "orders.txt"

orders = ["A101 coffee", "A102 tea", "A103 cake"]
for order in orders:
    with open(path, "w", encoding="utf-8") as f:
        f.write(order + "\n")

print(path.read_text(encoding="utf-8"), end="")
```

<details><summary><b>Hint</b></summary>

What does mode `"w"` do to a file every time it opens it? How many times is the file opened here?

</details>

<details><summary><b>Expected output</b></summary>

```text
A101 coffee
A102 tea
A103 cake
```

</details>

<details><summary><b>Solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "orders.txt"

orders = ["A101 coffee", "A102 tea", "A103 cake"]
with open(path, "w", encoding="utf-8") as f:
    for order in orders:
        f.write(order + "\n")

print(path.read_text(encoding="utf-8"), end="")
```

The `with` was inside the loop, so the file was opened in `"w"` mode three times, and each opening wiped what the previous pass wrote. Open the file **once**, outside the loop, and write every line inside it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "orders.txt"

orders = ["A101 coffee", "A102 tea", "A103 cake"]
with open(path, "w", encoding="utf-8") as f:
    f.writelines(f"{order}\n" for order in orders)

print(path.read_text(encoding="utf-8"), end="")
```

`writelines` writes every string it's given (it does *not* add newlines, despite the name). Changing `"w"` to `"a"` would also make the buggy version "work", but reopening a file for every line is slow, and a second run would duplicate every order.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "orders.txt"

orders = ["A101 coffee", "A102 tea", "A103 cake"]
for order in orders:
    with open(path, "w", encoding="utf-8") as f:
        f.write(order + "\n")

print(path.read_text(encoding="utf-8"), end="")


### NB13-Q19 · L3 · csv · Write

Write `products` to `path` with `csv.DictWriter`, with the columns in the
order `name,sku,price` and a header row. Print the file's text. Then read it
back with `csv.DictReader` and print the total of all prices.

```text
name,sku,price
Pen,P1,1.5
Notebook,P2,3.25
Stapler,P3,7.0
Total: 11.75
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

products = [
    {"sku": "P1", "name": "Pen", "price": 1.5},
    {"sku": "P2", "name": "Notebook", "price": 3.25},
    {"sku": "P3", "name": "Stapler", "price": 7.0},
]
path = folder / "products.csv"
```

<details><summary><b>Hint</b></summary>

`csv.DictWriter(f, fieldnames=[...])` decides the column order. Call `writeheader()` once, then `writerow(product)` for each dict. Prices come back as text.

</details>

<details><summary><b>Expected output</b></summary>

```text
name,sku,price
Pen,P1,1.5
Notebook,P2,3.25
Stapler,P3,7.0
Total: 11.75
```

</details>

<details><summary><b>Solution</b></summary>

```python
with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "sku", "price"])
    writer.writeheader()
    for product in products:
        writer.writerow(product)

print(path.read_text(encoding="utf-8"), end="")

total = 0
with open(path, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        total += float(row["price"])
print("Total:", total)
```

`fieldnames` sets the column order, whatever order the keys are in each dict. `writeheader()` writes the names as the first line. Reading back, every price is a string again, so `float()` is needed before adding.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
FIELDS = ["name", "sku", "price"]
with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(products)

print(path.read_text(encoding="utf-8"), end="")

with open(path, newline="", encoding="utf-8") as f:
    total = sum(float(row["price"]) for row in csv.DictReader(f))
print(f"Total: {total}")
```

`DictWriter` raises `ValueError` if a dict has a key that's not in `fieldnames`, which catches typos early. Pass `extrasaction="ignore"` when you *want* to drop extra keys. For money, seniors would use `Decimal` (NB16) instead of `float`.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

products = [
    {"sku": "P1", "name": "Pen", "price": 1.5},
    {"sku": "P2", "name": "Notebook", "price": 3.25},
    {"sku": "P3", "name": "Stapler", "price": 7.0},
]
path = folder / "products.csv"

# your code here


### NB13-Q20 · L3 · json · Error

An order has a set of tags. Which error does this raise?

**Which error does this raise?**

```python
import json

order = {"id": 7, "tags": {"gift", "fragile"}}
print(json.dumps(order))
```

<details><summary><b>Hint</b></summary>

Think about the list of types JSON knows about (Q06). Is a set one of them?

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError`: JSON has no set type, so `json.dumps` doesn't know how to write one ("Object of type set is not JSON serializable"). The fix is to convert it first, for example `sorted(order["tags"])`, which also gives a stable order.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import json

order = {"id": 7, "tags": {"gift", "fragile"}}
print(json.dumps({**order, "tags": sorted(order["tags"])}))
```

Seniors convert to JSON-friendly types at the edge, right before saving. `json.dumps(data, default=sorted)` is another option: `default` is called for any value JSON can't handle. `sorted` keeps the output the same every run, which a set wouldn't.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB13-Q21 · L3 · open() and with · Refactor

This code opens and closes files by hand with `try`/`finally`. Rewrite it
with `with`.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "hours.txt"

f = open(path, "w", encoding="utf-8")
try:
    f.write("Mon 7.5\n")
    f.write("Tue 8.0\n")
    f.write("Wed 6.5\n")
finally:
    f.close()

f = open(path, encoding="utf-8")
try:
    total = 0
    for line in f:
        day, hours = line.split()
        total += float(hours)
finally:
    f.close()

print("Hours worked:", total)
```

<details><summary><b>Hint</b></summary>

`with open(...) as f:` does the `try`/`finally`/`close()` for you.

</details>

<details><summary><b>Expected output</b></summary>

```text
Hours worked: 22.0
```

</details>

<details><summary><b>Solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "hours.txt"

with open(path, "w", encoding="utf-8") as f:
    f.write("Mon 7.5\n")
    f.write("Tue 8.0\n")
    f.write("Wed 6.5\n")

total = 0
with open(path, encoding="utf-8") as f:
    for line in f:
        day, hours = line.split()
        total += float(hours)

print("Hours worked:", total)
```

The `try`/`finally` pattern is exactly what `with` does behind the scenes: it closes the file when the block ends, whether the block finished normally or raised an error. Same safety, half the lines.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "hours.txt"

path.write_text("Mon 7.5\nTue 8.0\nWed 6.5\n", encoding="utf-8")

with open(path, encoding="utf-8") as f:
    total = sum(float(line.split()[1]) for line in f)

print("Hours worked:", total)
```

`write_text` covers the simple write. Inside the generator, `line.split()[1]` takes the hours column directly. Seniors only write `try`/`finally` by hand for resources that don't support `with`, and NB24 shows how to give your own objects `with` support.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "hours.txt"

f = open(path, "w", encoding="utf-8")
try:
    f.write("Mon 7.5\n")
    f.write("Tue 8.0\n")
    f.write("Wed 6.5\n")
finally:
    f.close()

f = open(path, encoding="utf-8")
try:
    total = 0
    for line in f:
        day, hours = line.split()
        total += float(hours)
finally:
    f.close()

print("Hours worked:", total)


### NB13-Q22 · L3 · configparser · Write

Read the settings file `path` with `configparser`. Print the host, the port
plus one (so it must be a number), whether debug is on (as a `bool`), and the
server's `timeout`, using `30` because the file doesn't set it. Finally,
print the section names.

```text
host: 127.0.0.1
next port: 8081
debug: True
timeout: 30
sections: ['server', 'database']
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import configparser

path = folder / "app.ini"
path.write_text(
    "[server]\nhost = 127.0.0.1\nport = 8080\ndebug = yes\n\n"
    "[database]\nname = shop.db\n",
    encoding="utf-8",
)
```

<details><summary><b>Hint</b></summary>

`config = configparser.ConfigParser()` then `config.read(path, encoding="utf-8")`. `config["server"]["host"]` gives text; `config.getint(...)` and `config.getboolean(...)` convert for you. `getint` takes `fallback=`.

</details>

<details><summary><b>Expected output</b></summary>

```text
host: 127.0.0.1
next port: 8081
debug: True
timeout: 30
sections: ['server', 'database']
```

</details>

<details><summary><b>Solution</b></summary>

```python
config = configparser.ConfigParser()
config.read(path, encoding="utf-8")

host = config["server"]["host"]
port = config.getint("server", "port")
debug = config.getboolean("server", "debug")
timeout = config.getint("server", "timeout", fallback=30)

print("host:", host)
print("next port:", port + 1)
print("debug:", debug)
print("timeout:", timeout)
print("sections:", config.sections())
```

INI files have `[sections]` of `key = value` lines, and every value is text. `getint` and `getboolean` convert it (`getboolean` accepts `yes`/`no`, `on`/`off`, `true`/`false`, `1`/`0`). `fallback=` is the value to use when the key is missing, instead of raising an error.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
config = configparser.ConfigParser()
config.read(path, encoding="utf-8")
server = config["server"]

print(f"host: {server['host']}")
print(f"next port: {server.getint('port') + 1}")
print(f"debug: {server.getboolean('debug')}")
print(f"timeout: {server.getint('timeout', fallback=30)}")
print(f"sections: {config.sections()}")
```

Naming the section once (`server = config["server"]`) avoids repeating `"server"` on every line. For new projects, seniors often prefer TOML (`tomllib`, NB31), which has real numbers and booleans, but INI files are everywhere in older tools.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import configparser

path = folder / "app.ini"
path.write_text(
    "[server]\nhost = 127.0.0.1\nport = 8080\ndebug = yes\n\n"
    "[database]\nname = shop.db\n",
    encoding="utf-8",
)

# your code here


### NB13-Q23 · L3 · xml · Write

Parse the XML text with `xml.etree.ElementTree`. Print each book's id,
title and year (as a number), then the title of the oldest book.

```text
b1 Dune 1965
b2 Foundation 1951
b3 Ubik 1969
Oldest: Foundation
```

**Starting code (already in the cell below):**

```python
import xml.etree.ElementTree as ET

text = """
<library>
  <book id="b1" year="1965"><title>Dune</title></book>
  <book id="b2" year="1951"><title>Foundation</title></book>
  <book id="b3" year="1969"><title>Ubik</title></book>
</library>
"""
```

<details><summary><b>Hint</b></summary>

`root = ET.fromstring(text)`. `root.findall("book")` gives the `<book>` elements. `book.get("year")` reads an attribute; `book.find("title").text` reads a child's text.

</details>

<details><summary><b>Expected output</b></summary>

```text
b1 Dune 1965
b2 Foundation 1951
b3 Ubik 1969
Oldest: Foundation
```

</details>

<details><summary><b>Solution</b></summary>

```python
root = ET.fromstring(text)
oldest_title = ""
oldest_year = 9999
for book in root.findall("book"):
    title = book.find("title").text
    year = int(book.get("year"))
    print(book.get("id"), title, year)
    if year < oldest_year:
        oldest_year = year
        oldest_title = title
print("Oldest:", oldest_title)
```

XML is a tree of elements. `fromstring` gives you the top element (`<library>`). Attributes like `year="1965"` are read with `.get()`, and they are always text. The text *between* tags, like `Dune`, is the element's `.text`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
root = ET.fromstring(text)
books = [(b.get("id"), b.findtext("title"), int(b.get("year"))) for b in root.iter("book")]
for book_id, title, year in books:
    print(book_id, title, year)
oldest = min(books, key=lambda book: book[2])
print("Oldest:", oldest[1])
```

`findtext` reads a child's text in one step. More important: the standard-library XML parsers are not safe against maliciously built XML. For XML from the internet, seniors use the `defusedxml` package, or better, ask for JSON.

</details>

In [ ]:
import xml.etree.ElementTree as ET

text = """
<library>
  <book id="b1" year="1965"><title>Dune</title></book>
  <book id="b2" year="1951"><title>Foundation</title></book>
  <book id="b3" year="1969"><title>Ubik</title></book>
</library>
"""

# your code here


### NB13-Q24 · L3 · OSError family · Predict

File errors all belong to the `OSError` family, and each carries an `errno`
code. Predict the output.

**What does this print?**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
missing = Path(tmp.name) / "nope.txt"

import errno

try:
    missing.read_text(encoding="utf-8")
except OSError as e:
    print(type(e).__name__)
    print(e.errno == errno.ENOENT)

print(type(OSError(errno.ENOENT, "No such file")).__name__)
print(type(OSError(errno.EACCES, "Permission denied")).__name__)
```

<details><summary><b>Hint</b></summary>

`except OSError` also catches its subclasses. Building an `OSError` with a known `errno` code gives you the matching subclass automatically.

</details>

<details><summary><b>Expected output</b></summary>

```text
FileNotFoundError
True
FileNotFoundError
PermissionError
```

</details>

<details><summary><b>Solution</b></summary>

A missing file raises `FileNotFoundError`, a subclass of `OSError`, so `except OSError` catches it. Its `errno` is `ENOENT` ("error: no entry"). Python maps codes to classes: build an `OSError` with `ENOENT` and you get a `FileNotFoundError`; with `EACCES` you get a `PermissionError`.

</details>

<details><summary><b>Senior dev solution</b></summary>

Seniors catch the specific subclass (`FileNotFoundError`, `PermissionError`, `IsADirectoryError`) rather than checking `e.errno` by hand. Catch plain `OSError` only when every file problem gets the same response, like "could not save, try again".

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB13-Q25 · L3 · csv · Refactor

This splits CSV lines by hand. Rewrite it with `csv.DictReader`, keeping the
output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "staff.csv"

path.write_text("name,dept,salary\nAda,R&D,5200\nLinus,Ops,4100\nGrace,R&D,6100\n",
                encoding="utf-8")

lines = path.read_text(encoding="utf-8").split("\n")
header = lines[0].split(",")
total = 0
count = 0
for i in range(1, len(lines)):
    if lines[i] == "":
        continue
    values = lines[i].split(",")
    row = {}
    for j in range(len(header)):
        row[header[j]] = values[j]
    if row["dept"] == "R&D":
        total = total + int(row["salary"])
        count = count + 1
print("R&D average:", total / count)
```

<details><summary><b>Hint</b></summary>

`csv.DictReader(f)` already turns each line into a dict keyed by the header, and skips the empty end of the file.

</details>

<details><summary><b>Expected output</b></summary>

```text
R&D average: 5650.0
```

</details>

<details><summary><b>Solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "staff.csv"

import csv

path.write_text("name,dept,salary\nAda,R&D,5200\nLinus,Ops,4100\nGrace,R&D,6100\n",
                encoding="utf-8")

total = 0
count = 0
with open(path, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["dept"] == "R&D":
            total += int(row["salary"])
            count += 1
print("R&D average:", total / count)
```

`DictReader` replaces the header split, the index loop and the hand-built dict. It also handles quoted values like `"Smith, John"`, which `split(",")` would cut in two. `+=` replaces `total = total + ...`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "staff.csv"

import csv

path.write_text("name,dept,salary\nAda,R&D,5200\nLinus,Ops,4100\nGrace,R&D,6100\n",
                encoding="utf-8")

with open(path, newline="", encoding="utf-8") as f:
    salaries = [int(row["salary"]) for row in csv.DictReader(f) if row["dept"] == "R&D"]
print("R&D average:", sum(salaries) / len(salaries))
```

Collecting the matching salaries in a list keeps the total and the count in step automatically. Seniors never parse CSV with `split(",")`: the first address or name with a comma in it breaks it silently.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
path = Path(tmp.name) / "staff.csv"

path.write_text("name,dept,salary\nAda,R&D,5200\nLinus,Ops,4100\nGrace,R&D,6100\n",
                encoding="utf-8")

lines = path.read_text(encoding="utf-8").split("\n")
header = lines[0].split(",")
total = 0
count = 0
for i in range(1, len(lines)):
    if lines[i] == "":
        continue
    values = lines[i].split(",")
    row = {}
    for j in range(len(header)):
        row[header[j]] = values[j]
    if row["dept"] == "R&D":
        total = total + int(row["salary"])
        count = count + 1
print("R&D average:", total / count)


---

## L4 · Somewhat Difficult

### NB13-Q26 · L4 · Your own module · Write

Turn `module_code` into a real module and watch `__name__` change.

1. Write `module_code` to `folder / "shout_tools.py"`.
2. Add `folder` to `sys.path`, import the module, and print
   `shout_tools.shout("hello")`. You can write `import shout_tools`, or
   `shout_tools = importlib.import_module("shout_tools")`, which does the
   same job with the name as a string.
3. Print `__name__` from your own cell, in the format shown.
4. Run the file as a script with `runpy.run_path(str(path), run_name="__main__")`.

```text
shout_tools sees __name__ = shout_tools
HELLO!
this cell sees __name__ = __main__
shout_tools sees __name__ = __main__
shout_tools was run directly
```

(Run the cell twice and the first line disappears. The explanation says why.)

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import importlib
import runpy
import sys

module_code = """
def shout(text):
    return text.upper() + "!"

print("shout_tools sees __name__ =", __name__)

if __name__ == "__main__":
    print("shout_tools was run directly")
"""
```

<details><summary><b>Hint</b></summary>

`path.write_text(module_code, encoding="utf-8")` creates the file. `sys.path.insert(0, str(folder))` tells `import` to look in `folder` first.

</details>

<details><summary><b>Expected output</b></summary>

```text
shout_tools sees __name__ = shout_tools
HELLO!
this cell sees __name__ = __main__
shout_tools sees __name__ = __main__
shout_tools was run directly
```

</details>

<details><summary><b>Solution</b></summary>

```python
path = folder / "shout_tools.py"
path.write_text(module_code, encoding="utf-8")

sys.path.insert(0, str(folder))
shout_tools = importlib.import_module("shout_tools")

print(shout_tools.shout("hello"))
print("this cell sees __name__ =", __name__)
runpy.run_path(str(path), run_name="__main__")
```

Any `.py` file is a module. `import` searches the folders in `sys.path`, runs the file's top-level code once, and keeps the result. While it runs, `__name__` is the module's name, so the `if` block is skipped. Code run directly (your cell, or a script) has `__name__ == "__main__"`, so the block runs. Python caches imported modules in `sys.modules`, which is why a second `import` (rerunning the cell) doesn't print again.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
path = folder / "shout_tools.py"
path.write_text(module_code, encoding="utf-8")

sys.path.insert(0, str(folder))
import shout_tools

print(shout_tools.shout("hello"))
print(f"this cell sees __name__ = {__name__}")
runpy.run_path(str(path), run_name="__main__")
```

When the module name is fixed, seniors write the plain `import` statement: editors and linters understand it. `import_module` is for names only known while the program runs, like plugins listed in a config file. `if __name__ == "__main__":` lets one file be both a library and a script. Seniors keep module top-level code free of `print` and other side effects (this one prints only to show you what happens), and put the script part in a `main()` function called from that `if`.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import importlib
import runpy
import sys

module_code = """
def shout(text):
    return text.upper() + "!"

print("shout_tools sees __name__ =", __name__)

if __name__ == "__main__":
    print("shout_tools was run directly")
"""

# your code here


### NB13-Q27 · L4 · Packages · Write

Build a package called `corner_shop` inside `folder`: a sub-folder holding
three files, `__init__.py`, `prices.py` and `cart.py`, with the given texts.
Add `folder` to `sys.path` and import the package. Then print
`corner_shop.cart_total([10, 5.5])` and the `TAX_RATE` from the `prices`
module. (`import corner_shop` and `from corner_shop.prices import TAX_RATE`
work, and so does `importlib.import_module("corner_shop.prices")`.)

```text
18.6
0.2
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import sys

import importlib

init_code = "from .cart import cart_total\n"
prices_code = "TAX_RATE = 0.2\n\ndef with_tax(amount):\n    return round(amount * (1 + TAX_RATE), 2)\n"
cart_code = "from .prices import with_tax\n\ndef cart_total(amounts):\n    return with_tax(sum(amounts))\n"
```

<details><summary><b>Hint</b></summary>

`package = folder / "corner_shop"`, then `package.mkdir()`, then one `write_text` per file. The dots in `from .prices import ...` mean "from this same package".

</details>

<details><summary><b>Expected output</b></summary>

```text
18.6
0.2
```

</details>

<details><summary><b>Solution</b></summary>

```python
package = folder / "corner_shop"
package.mkdir()
(package / "__init__.py").write_text(init_code, encoding="utf-8")
(package / "prices.py").write_text(prices_code, encoding="utf-8")
(package / "cart.py").write_text(cart_code, encoding="utf-8")

sys.path.insert(0, str(folder))
corner_shop = importlib.import_module("corner_shop")
prices = importlib.import_module("corner_shop.prices")

print(corner_shop.cart_total([10, 5.5]))
print(prices.TAX_RATE)
```

A *package* is a folder of modules with an `__init__.py`, which runs when the package is imported. Here it pulls `cart_total` up to the package level, so `corner_shop.cart_total` works. Inside the package, `from .prices import with_tax` is a *relative import*: the dot means "this package", so the code keeps working if the package is renamed.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
package = folder / "corner_shop"
package.mkdir()
files = {"__init__.py": init_code, "prices.py": prices_code, "cart.py": cart_code}
for name, code in files.items():
    (package / name).write_text(code, encoding="utf-8")

sys.path.insert(0, str(folder))
import corner_shop
from corner_shop.prices import TAX_RATE

print(corner_shop.cart_total([10, 5.5]))
print(TAX_RATE)
```

A dict of file names to contents removes the copy-paste. Relative imports only work *inside* a package; running `cart.py` directly would fail with `ImportError`, which is why NB31 runs packages with `python -m`. Real projects install packages with `pip install -e .` instead of editing `sys.path`.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import sys

import importlib

init_code = "from .cart import cart_total\n"
prices_code = "TAX_RATE = 0.2\n\ndef with_tax(amount):\n    return round(amount * (1 + TAX_RATE), 2)\n"
cart_code = "from .prices import with_tax\n\ndef cart_total(amounts):\n    return with_tax(sum(amounts))\n"

# your code here


### NB13-Q28 · L4 · Relative paths · Fix

The CSV is created correctly, but reading it raises `FileNotFoundError`. Fix
the program so it prints `Total visits: 215`.

**Buggy code (copied into the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

(folder / "visits_2024.csv").write_text("day,visits\nMon,120\nTue,95\n", encoding="utf-8")

total = 0
with open("visits_2024.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        total += int(row["visits"])
print("Total visits:", total)
```

<details><summary><b>Hint</b></summary>

Where does Python look for a bare file name like `"visits_2024.csv"`? Is that where the file was written?

</details>

<details><summary><b>Expected output</b></summary>

```text
Total visits: 215
```

</details>

<details><summary><b>Solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

(folder / "visits_2024.csv").write_text("day,visits\nMon,120\nTue,95\n", encoding="utf-8")

total = 0
with open(folder / "visits_2024.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        total += int(row["visits"])
print("Total visits:", total)
```

A bare name like `"visits_2024.csv"` is a *relative path*: Python looks for it in the **current working directory** (`os.getcwd()`), not in the folder where you wrote the file. Joining it onto `folder` gives the full path, which works wherever the program is started from.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

visits_path = folder / "visits_2024.csv"
visits_path.write_text("day,visits\nMon,120\nTue,95\n", encoding="utf-8")

with open(visits_path, newline="", encoding="utf-8") as f:
    total = sum(int(row["visits"]) for row in csv.DictReader(f))
print("Total visits:", total)
```

Build the path once, name it, and use the name for both the write and the read, so they can't drift apart. In scripts, seniors anchor data files to the script's own location with `Path(__file__).parent / "data.csv"`, because the working directory depends on how the program was started.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

(folder / "visits_2024.csv").write_text("day,visits\nMon,120\nTue,95\n", encoding="utf-8")

total = 0
with open("visits_2024.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        total += int(row["visits"])
print("Total visits:", total)


### NB13-Q29 · L4 · json · Fix

Room bookings are saved to JSON and loaded back, then room `102` is looked up.
It raises `KeyError`. Fix it so the loaded dict has **int** keys again and the
program prints `Room 102 is booked by Linus`.

**Buggy code (copied into the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import json

bookings = {101: "Ada", 102: "Linus", 103: "Grace"}
path = folder / "bookings.json"
path.write_text(json.dumps(bookings), encoding="utf-8")

loaded = json.loads(path.read_text(encoding="utf-8"))
room = 102
print("Room", room, "is booked by", loaded[room])
```

<details><summary><b>Hint</b></summary>

Print `loaded` and look closely at the keys. Q06 showed what JSON does to them.

</details>

<details><summary><b>Expected output</b></summary>

```text
Room 102 is booked by Linus
```

</details>

<details><summary><b>Solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import json

bookings = {101: "Ada", 102: "Linus", 103: "Grace"}
path = folder / "bookings.json"
path.write_text(json.dumps(bookings), encoding="utf-8")

raw = json.loads(path.read_text(encoding="utf-8"))
loaded = {}
for key, name in raw.items():
    loaded[int(key)] = name
room = 102
print("Room", room, "is booked by", loaded[room])
```

JSON object keys are always strings, so `102` was saved as `"102"` and `loaded[102]` finds nothing. Rebuilding the dict with `int(key)` restores the original keys, so every later lookup works with real room numbers.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import json

bookings = {101: "Ada", 102: "Linus", 103: "Grace"}
path = folder / "bookings.json"
path.write_text(json.dumps(bookings), encoding="utf-8")

loaded = {int(room): name for room, name in json.loads(path.read_text(encoding="utf-8")).items()}
room = 102
print(f"Room {room} is booked by {loaded[room]}")
```

A dict comprehension (NB10) converts the keys in one line, right where the data is loaded. Patching individual lookups with `loaded[str(room)]` instead would leave string keys spread through the program, waiting to bite someone else.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import json

bookings = {101: "Ada", 102: "Linus", 103: "Grace"}
path = folder / "bookings.json"
path.write_text(json.dumps(bookings), encoding="utf-8")

loaded = json.loads(path.read_text(encoding="utf-8"))
room = 102
print("Room", room, "is booked by", loaded[room])


### NB13-Q30 · L4 · pickle · Write

`shifts` has tuple keys, which JSON can't store. Save it with `pickle.dump`
(mode `"wb"`), load it back with `pickle.load` (mode `"rb"`), and print
whether the copy equals the original, then the Tuesday 14:00 shift. Finally,
try `json.dumps(shifts)` and print which error it raises.

```text
True
['Grace']
JSON failed: TypeError
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import json
import pickle

shifts = {("Mon", 9): ["Ada", "Linus"], ("Tue", 14): ["Grace"]}
path = folder / "shifts.pkl"
```

<details><summary><b>Hint</b></summary>

Pickle files are binary, so open them with `"wb"` and `"rb"` and no `encoding=`. For the last part, catch the error `as e` and print `type(e).__name__`.

</details>

<details><summary><b>Expected output</b></summary>

```text
True
['Grace']
JSON failed: TypeError
```

</details>

<details><summary><b>Solution</b></summary>

```python
with open(path, "wb") as f:
    pickle.dump(shifts, f)

with open(path, "rb") as f:
    loaded = pickle.load(f)

print(loaded == shifts)
print(loaded[("Tue", 14)])

try:
    json.dumps(shifts)
except TypeError as e:
    print("JSON failed:", type(e).__name__)
```

`pickle` saves almost any Python object exactly: tuple keys stay tuples, sets stay sets. The file is binary (bytes, not text), hence `"b"` in the mode. JSON refuses non-string keys with `TypeError`.

**Never unpickle data you don't trust.** Loading a pickle can run any code the file's author chose. Only load pickles your own program wrote.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
path.write_bytes(pickle.dumps(shifts))
loaded = pickle.loads(path.read_bytes())

print(loaded == shifts)
print(loaded[("Tue", 14)])

try:
    json.dumps(shifts)
except TypeError as e:
    print(f"JSON failed: {type(e).__name__}")
```

`write_bytes`/`read_bytes` with `dumps`/`loads` are the short versions. Seniors use pickle for caches their own program creates and reads, and JSON for anything shared with other programs or people: it's readable, works in every language, and loading it can't run code.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import json
import pickle

shifts = {("Mon", 9): ["Ada", "Linus"], ("Tue", 14): ["Grace"]}
path = folder / "shifts.pkl"

# your code here


### NB13-Q31 · L4 · import * · Predict

`json` declares its public names in `__all__`. Import it with `*` and check
which names arrived. Predict the output.

**What does this print?**

```python
from json import *

names = dir()
print("dumps" in names, "JSONDecodeError" in names)
print("detect_encoding" in names, "_default_encoder" in names)

import json
print(json.__all__)
print("detect_encoding" in dir(json), "_default_encoder" in dir(json))
```

<details><summary><b>Hint</b></summary>

`dir()` with no argument lists the names you can use right here. `import *` copies only the names listed in the module's `__all__`, if it has one. `dir(module)` lists everything inside the module.

</details>

<details><summary><b>Expected output</b></summary>

```text
True True
False False
['dump', 'dumps', 'load', 'loads', 'JSONDecoder', 'JSONDecodeError', 'JSONEncoder']
True True
```

</details>

<details><summary><b>Solution</b></summary>

`from json import *` copies exactly the names in `json.__all__`, so `dumps` and `JSONDecodeError` arrive but `detect_encoding` (a real function in `json`) does not. Without `__all__`, `*` copies every name that doesn't start with `_`, so `_default_encoder` would still stay behind. `dir(json)` looks *inside* the module, where everything exists. Your own modules (Q26) can set `__all__ = ["shout"]` the same way.

</details>

<details><summary><b>Senior dev solution</b></summary>

PEP 8 says to avoid `import *`: readers can't tell where a name came from, and a later `*` import can silently replace an earlier name. It's fine in an interactive session. Library authors still set `__all__` to declare their public names.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB13-Q32 · L4 · __pycache__ · Predict

What does importing a module leave behind in its folder? Predict the output.
(The `sys.dont_write_bytecode` line makes sure caching is switched on; some
setups turn it off. Predict for the *first* run: a second run reuses the
module already imported, as Q26 explains.)

**What does this print?**

```python
import importlib
import sys
import tempfile
from pathlib import Path

sys.dont_write_bytecode = False
tmp = tempfile.TemporaryDirectory()
folder = Path(tmp.name)
(folder / "tax_tools.py").write_text("RATE = 0.2\n", encoding="utf-8")
sys.path.insert(0, str(folder))

cache = folder / "__pycache__"
print(cache.exists())
tax_tools = importlib.import_module("tax_tools")  # same as: import tax_tools
print(cache.exists())
print(len(list(cache.glob("tax_tools.*.pyc"))))
print(tax_tools.RATE)
```

<details><summary><b>Hint</b></summary>

Python compiles a module to *bytecode* the first time it imports it, and saves that so the next import is faster.

</details>

<details><summary><b>Expected output</b></summary>

```text
False
True
1
0.2
```

</details>

<details><summary><b>Solution</b></summary>

Importing `tax_tools` compiles it to bytecode and saves it as `__pycache__/tax_tools.cpython-3XX.pyc` (the version number is in the name, so several Pythons can share a folder). Next time, if the `.py` file hasn't changed, Python loads the `.pyc` and skips compiling. Your cell itself is not imported, so it never gets a `.pyc`.

</details>

<details><summary><b>Senior dev solution</b></summary>

`__pycache__` folders are safe to delete and are recreated as needed. Seniors add `__pycache__/` to `.gitignore` so compiled files never end up in version control.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB13-Q33 · L4 · pathlib · Refactor

This uses `os.path` and `glob` with string paths. Rewrite it with `pathlib`
and `with` (or `write_text`), keeping the output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
import glob
import os
import tempfile

tmp = tempfile.TemporaryDirectory()
base = tmp.name
os.makedirs(os.path.join(base, "reports"))
for name in ["jan.csv", "feb.csv", "notes.md"]:
    f = open(os.path.join(base, "reports", name), "w", encoding="utf-8")
    f.write("month,total")
    f.close()

for full in sorted(glob.glob(os.path.join(base, "reports", "*.csv"))):
    file_name = os.path.basename(full)
    stem = os.path.splitext(file_name)[0]
    print(stem, os.path.getsize(full))
```

<details><summary><b>Hint</b></summary>

`reports = Path(tmp.name) / "reports"` and `reports.mkdir()`. A `Path` has `.glob()`, `.stem` and `.stat().st_size`.

</details>

<details><summary><b>Expected output</b></summary>

```text
feb 11
jan 11
```

</details>

<details><summary><b>Solution</b></summary>

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
reports = Path(tmp.name) / "reports"
reports.mkdir()
for name in ["jan.csv", "feb.csv", "notes.md"]:
    (reports / name).write_text("month,total", encoding="utf-8")

for path in sorted(reports.glob("*.csv")):
    print(path.stem, path.stat().st_size)
```

`Path` objects carry their own tools: `/` instead of `os.path.join`, `.glob()` instead of `glob.glob`, `.stem` instead of `splitext(basename(...))[0]`, and `.stat().st_size` instead of `os.path.getsize`. `write_text` replaces the open, write, close trio.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    reports = Path(tmp) / "reports"
    reports.mkdir()
    for name in ["jan.csv", "feb.csv", "notes.md"]:
        (reports / name).write_text("month,total", encoding="utf-8")

    for path in sorted(reports.glob("*.csv")):
        print(path.stem, path.stat().st_size)
```

The `with` form deletes the folder the moment the work is done. You'll still see `os.path` in older code, and `os` remains the place for environment variables (`os.environ`, NB31) and process details, but for paths, seniors reach for `pathlib`.

</details>

In [ ]:
import glob
import os
import tempfile

tmp = tempfile.TemporaryDirectory()
base = tmp.name
os.makedirs(os.path.join(base, "reports"))
for name in ["jan.csv", "feb.csv", "notes.md"]:
    f = open(os.path.join(base, "reports", name), "w", encoding="utf-8")
    f.write("month,total")
    f.close()

for full in sorted(glob.glob(os.path.join(base, "reports", "*.csv"))):
    file_name = os.path.basename(full)
    stem = os.path.splitext(file_name)[0]
    print(stem, os.path.getsize(full))


---

## L5 · Hard

### NB13-Q34 · L5 · Log analyser · Write

Analyse `access.log` line by line. A valid line has exactly 6 fields:
`date time method path status seconds`. Ignore blank lines without counting
them; count any other line without 6 fields as `skipped`.

Build a summary dict with:
- `requests`: the number of valid lines
- `skipped`: the number of bad lines
- `status_counts`: how many times each status code appeared (as an `int` key)
- `slowest_path` and `slowest_seconds`: the slowest request

Save it to `summary_path` with `json.dump(..., indent=2, sort_keys=True)`,
print the file's text, then load the file back and print the number of `200`
responses from the loaded data.

```text
{
  "requests": 6,
  "skipped": 1,
  "slowest_path": "/cart",
  "slowest_seconds": 1.93,
  "status_counts": {
    "200": 3,
    "201": 1,
    "404": 1,
    "500": 1
  }
}
200 responses: 3
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import json

log_path = folder / "access.log"
log_path.write_text(
    "2024-05-01 09:00:01 GET /home 200 0.12\n"
    "2024-05-01 09:00:03 GET /shop 200 0.48\n"
    "2024-05-01 09:00:04 POST /cart 500 1.93\n"
    "\n"
    "2024-05-01 09:00:09 GET /missing 404 0.05\n"
    "garbled line\n"
    "2024-05-01 09:00:12 GET /shop 200 0.51\n"
    "2024-05-01 09:00:15 POST /cart 201 0.33\n",
    encoding="utf-8",
)
summary_path = folder / "summary.json"
```

<details><summary><b>Hint</b></summary>

Strip each line; skip it if it's empty. Split it; if there aren't 6 parts, add one to `skipped`. Count statuses with `counts.get(status, 0) + 1`. After loading the JSON back, look at the keys of `status_counts` before you index into it.

</details>

<details><summary><b>Expected output</b></summary>

```text
{
  "requests": 6,
  "skipped": 1,
  "slowest_path": "/cart",
  "slowest_seconds": 1.93,
  "status_counts": {
    "200": 3,
    "201": 1,
    "404": 1,
    "500": 1
  }
}
200 responses: 3
```

</details>

<details><summary><b>Solution</b></summary>

```python
requests = 0
skipped = 0
status_counts = {}
slowest_path = ""
slowest_seconds = 0.0

with open(log_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 6:
            skipped += 1
            continue
        date, time, method, path, status, seconds = parts
        requests += 1
        status = int(status)
        status_counts[status] = status_counts.get(status, 0) + 1
        if float(seconds) > slowest_seconds:
            slowest_seconds = float(seconds)
            slowest_path = path

summary = {
    "requests": requests,
    "skipped": skipped,
    "status_counts": status_counts,
    "slowest_path": slowest_path,
    "slowest_seconds": slowest_seconds,
}
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, sort_keys=True)

print(summary_path.read_text(encoding="utf-8"))

with open(summary_path, encoding="utf-8") as f:
    loaded = json.load(f)
print("200 responses:", loaded["status_counts"]["200"])
```

Processing the file line by line keeps memory flat however big the log grows. Blank lines are skipped before counting, so only genuinely broken lines count as `skipped`. Unpacking the six parts into names makes the rest readable.

The trap is at the end: `status_counts` had `int` keys in memory, but JSON keys are always strings, so the loaded data needs `["200"]`, not `[200]`. `sort_keys=True` sorts the keys (here `"200"` before `"201"`), so the file is the same every run, which makes it easy to compare.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def parse_log(path):
    summary = {"requests": 0, "skipped": 0, "status_counts": {},
               "slowest_path": "", "slowest_seconds": 0.0}
    counts = summary["status_counts"]
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                _date, _time, _method, url, status, seconds = line.split()
            except ValueError:
                summary["skipped"] += 1
                continue
            summary["requests"] += 1
            counts[int(status)] = counts.get(int(status), 0) + 1
            if float(seconds) > summary["slowest_seconds"]:
                summary["slowest_path"], summary["slowest_seconds"] = url, float(seconds)
    return summary


summary_path.write_text(json.dumps(parse_log(log_path), indent=2, sort_keys=True),
                        encoding="utf-8")
print(summary_path.read_text(encoding="utf-8"))
loaded = json.loads(summary_path.read_text(encoding="utf-8"))
print(f"200 responses: {loaded['status_counts']['200']}")
```

Seniors wrap the parsing in a function so it can be tested on its own, and let unpacking do the validation: a line with the wrong number of fields raises `ValueError` (EAFP). Unused fields get a leading `_`. `collections.Counter` (NB15) would replace the `.get(..., 0) + 1` counting.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import json

log_path = folder / "access.log"
log_path.write_text(
    "2024-05-01 09:00:01 GET /home 200 0.12\n"
    "2024-05-01 09:00:03 GET /shop 200 0.48\n"
    "2024-05-01 09:00:04 POST /cart 500 1.93\n"
    "\n"
    "2024-05-01 09:00:09 GET /missing 404 0.05\n"
    "garbled line\n"
    "2024-05-01 09:00:12 GET /shop 200 0.51\n"
    "2024-05-01 09:00:15 POST /cart 201 0.33\n",
    encoding="utf-8",
)
summary_path = folder / "summary.json"

# your code here


### NB13-Q35 · L5 · shelve · Predict

`shelve` is a dict that lives in a file. Watch what happens when you change a
list stored inside it. Predict the output.

**What does this print?**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import shelve

db_path = str(folder / "stock")

with shelve.open(db_path) as db:
    db["fruit"] = ["apple"]
    db["fruit"].append("pear")
    print(db["fruit"])

    items = db["fruit"]
    items.append("plum")
    db["fruit"] = items

with shelve.open(db_path) as db:
    print(db["fruit"])
    print(list(db.keys()))
```

<details><summary><b>Hint</b></summary>

Every time you read `db["fruit"]`, shelve unpickles a **fresh copy** from the file. What happens to a change made to a copy that is never saved?

</details>

<details><summary><b>Expected output</b></summary>

```text
['apple']
['apple', 'plum']
['fruit']
```

</details>

<details><summary><b>Solution</b></summary>

`db["fruit"].append("pear")` reads a copy of the list, appends to the copy, and throws the copy away: the file never hears about `pear`. The second block does it properly: read into `items`, change it, and assign it back, which writes it to the file. Reopening the shelf proves the data survived: `['apple', 'plum']`.

</details>

<details><summary><b>Senior dev solution</b></summary>

`shelve.open(path, writeback=True)` caches values so in-place changes are saved on close, at the cost of memory. Seniors prefer the explicit read-change-assign pattern. Shelve uses pickle underneath, so the same rule applies: never open a shelf someone else gave you.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB13-Q36 · L5 · csv · Predict

This simulates what happens on **Windows** when you forget `newline=""`.
Opening with `newline="\r\n"` makes Python turn every `\n` into `\r\n` on
writing, which is exactly what Windows does by default. Predict the output.

**What does this print?**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv

path = folder / "scores.csv"
with open(path, "w", newline="\r\n", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "score"])
    writer.writerow(["Ada", 90])

with open(path, newline="", encoding="utf-8") as f:
    print(repr(f.read()))

with open(path, newline="", encoding="utf-8") as f:
    print(list(csv.reader(f)))
```

<details><summary><b>Hint</b></summary>

`csv.writer` already ends each row with `\r\n`. Then the file translates the `\n` in that ending. Reading with `newline=""` shows the characters exactly as stored.

</details>

<details><summary><b>Expected output</b></summary>

```text
'name,score\r\r\nAda,90\r\r\n'
[['name', 'score'], [], ['Ada', '90'], []]
```

</details>

<details><summary><b>Solution</b></summary>

The csv module ends rows with `\r\n` itself. The file then turns the `\n` into `\r\n`, so each row really ends with `\r\r\n`. A CSV reader sees `\r` as one line end and `\r\n` as another, so every row is followed by an empty row `[]`. On Windows this shows up as blank lines between rows in Excel. `newline=""` turns the translation off and lets csv handle line endings itself, which is why every csv example opens files that way.

</details>

<details><summary><b>Senior dev solution</b></summary>

On Linux and macOS, forgetting `newline=""` looks harmless, so the bug only appears when a colleague runs your code on Windows. Seniors write `newline=""` every time they open a file for the csv module, reading or writing, so the code behaves the same everywhere.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## Mini-project

Pull together everything from this notebook.

## NB13-P · L5 · Mini-project: Contacts Manager · Write

You have an address book saved as JSON, and a CSV of new contacts to import.

1. Load the contacts from `contacts_path` (a JSON list of dicts).
2. Read `import_path` with `csv.DictReader`. Clean each email with
   `.strip().lower()`. Skip a row if its email is empty (count it as
   *missing email*) or if that email is already in the address book,
   including ones added earlier in this import (count it as *duplicate*).
   Otherwise add `{"name": ..., "email": ..., "phone": ...}` to the contacts.
3. Sort the contacts by name and save them back to `contacts_path` with
   `indent=2`.
4. Write `report_path` as a CSV with the columns `name,email`, in the same
   order.
5. Print the summary, how many contacts are in the JSON file when you load
   it again, and the report's text:

```text
Added: 2
Duplicates skipped: 2
Missing email skipped: 1
Reloaded from JSON: 4
--- report.csv ---
name,email
Ada Lovelace,ada@example.com
Alan Turing,alan@example.com
Grace Hopper,grace@example.com
Margaret Hamilton,margaret@example.com
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv
import json

contacts_path = folder / "contacts.json"
contacts_path.write_text(json.dumps([
    {"name": "Alan Turing", "email": "alan@example.com", "phone": "555-0102"},
    {"name": "Ada Lovelace", "email": "ada@example.com", "phone": "555-0101"},
]), encoding="utf-8")

import_path = folder / "new_contacts.csv"
import_path.write_text(
    "name,email,phone\n"
    "Grace Hopper,grace@example.com,555-0103\n"
    "Ada L.,ADA@example.com ,555-0199\n"
    "Linus Torvalds,,555-0104\n"
    "Margaret Hamilton,margaret@example.com,555-0105\n"
    "Grace Hopper,grace@example.com,555-0103\n",
    encoding="utf-8",
)
report_path = folder / "report.csv"
```

<details><summary><b>Hint</b></summary>

Keep a set of known emails, built from the loaded contacts and updated as you add each new one, so both kinds of duplicate are caught. Sort with `key=lambda c: c["name"]`. `DictWriter` with `extrasaction="ignore"` leaves out the `phone` key.

</details>

<details><summary><b>Expected output</b></summary>

```text
Added: 2
Duplicates skipped: 2
Missing email skipped: 1
Reloaded from JSON: 4
--- report.csv ---
name,email
Ada Lovelace,ada@example.com
Alan Turing,alan@example.com
Grace Hopper,grace@example.com
Margaret Hamilton,margaret@example.com
```

</details>

<details><summary><b>Solution</b></summary>

```python
def load_contacts(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def import_contacts(path, contacts):
    known = {contact["email"] for contact in contacts}
    added = duplicates = missing = 0
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            email = row["email"].strip().lower()
            if not email:
                missing += 1
            elif email in known:
                duplicates += 1
            else:
                contacts.append({"name": row["name"], "email": email, "phone": row["phone"]})
                known.add(email)
                added += 1
    return added, duplicates, missing


def save_contacts(path, contacts):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(contacts, f, indent=2)


def write_report(path, contacts):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["name", "email"], extrasaction="ignore")
        writer.writeheader()
        for contact in contacts:
            writer.writerow(contact)


contacts = load_contacts(contacts_path)
added, duplicates, missing = import_contacts(import_path, contacts)
contacts.sort(key=lambda contact: contact["name"])
save_contacts(contacts_path, contacts)
write_report(report_path, contacts)

print("Added:", added)
print("Duplicates skipped:", duplicates)
print("Missing email skipped:", missing)
print("Reloaded from JSON:", len(load_contacts(contacts_path)))
print("--- report.csv ---")
print(report_path.read_text(encoding="utf-8"), end="")
```

Each step is its own small function, so each can be checked alone. The `known` set makes the duplicate check fast and catches both kinds: `ADA@example.com ` matches Ada after cleaning, and the second Grace matches the Grace added a moment earlier because `known.add(email)` runs as soon as she's added. `if not email` catches the empty email before the duplicate check. Sorting once, before saving, means the JSON file and the report agree. Reloading the JSON proves the save really worked.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def load_contacts(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except FileNotFoundError:
        return []


def import_contacts(path, contacts):
    known = {c["email"] for c in contacts}
    stats = {"added": 0, "duplicates": 0, "missing": 0}
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            email = (row.get("email") or "").strip().lower()
            if not email:
                stats["missing"] += 1
                continue
            if email in known:
                stats["duplicates"] += 1
                continue
            contacts.append({"name": row["name"].strip(), "email": email,
                             "phone": row["phone"].strip()})
            known.add(email)
            stats["added"] += 1
    return stats


def save_contacts(path, contacts):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(contacts, indent=2), encoding="utf-8")
    temp.replace(path)


def write_report(path, contacts):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["name", "email"], extrasaction="ignore")
        writer.writeheader()
        writer.writerows(contacts)


contacts = load_contacts(contacts_path)
stats = import_contacts(import_path, contacts)
contacts.sort(key=lambda c: c["name"].casefold())
save_contacts(contacts_path, contacts)
write_report(report_path, contacts)

print(f"Added: {stats['added']}")
print(f"Duplicates skipped: {stats['duplicates']}")
print(f"Missing email skipped: {stats['missing']}")
print(f"Reloaded from JSON: {len(load_contacts(contacts_path))}")
print("--- report.csv ---")
print(report_path.read_text(encoding="utf-8"), end="")
```

Three senior habits here. `load_contacts` returns an empty book on the first run instead of crashing. `save_contacts` writes to a temporary file and then *replaces* the real one, so a crash halfway through can never leave a half-written address book (an *atomic* save). And sorting with `casefold()` puts `adam` and `Ada` where a human expects. A `collections.Counter` (NB15) or a small class (NB14) could hold the stats.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()  # a throwaway folder
folder = Path(tmp.name)

import csv
import json

contacts_path = folder / "contacts.json"
contacts_path.write_text(json.dumps([
    {"name": "Alan Turing", "email": "alan@example.com", "phone": "555-0102"},
    {"name": "Ada Lovelace", "email": "ada@example.com", "phone": "555-0101"},
]), encoding="utf-8")

import_path = folder / "new_contacts.csv"
import_path.write_text(
    "name,email,phone\n"
    "Grace Hopper,grace@example.com,555-0103\n"
    "Ada L.,ADA@example.com ,555-0199\n"
    "Linus Torvalds,,555-0104\n"
    "Margaret Hamilton,margaret@example.com,555-0105\n"
    "Grace Hopper,grace@example.com,555-0103\n",
    encoding="utf-8",
)
report_path = folder / "report.csv"

# your code here


---

## Done

Next up: **NB14 · OOP Basics** in `intermediate/`.